II. Dataset a limpiar: personas desaparecidas en México.

Indicar:
- Número de filas
- Número de columnas
- Periodo de fechas cubierto
- Unidad de análisis (P.E, cada fila representa un registro, reporte o caso, según corresponda al dataset asignado)

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

dataset_path = "tot_reg_desap.csv"

df = pd.read_csv(dataset_path)  # lectura del dataset

print(f"Filas: {df.shape[0]}")
print(f"Columnas: {df.shape[1]}")

print(df.columns.to_list())  # ver las categorias de las columnas

# Periodo de fechas (1. ordenamos respecto a la fecha de desaparición, 2. Seleccionamos la primera y última)
df_ord_fechas = df.sort_values(by="fecha_desap").dropna()  # quitamos los NaN

primero_y_ultimo = df_ord_fechas.iloc[[0, -1]]
print("\nPrimera Fecha y última fecha:")
print(primero_y_ultimo["fecha_desap"])

# Imprimir head para analizar el significado de las entradas del dataset
df.head()

Filas: 89720
Columnas: 13
['Unnamed: 0', 'nombrecompleto', 'rnpdno_json', 'entidad_desap', 'pre_rnped', 'rnped', 'rnpdno_csv', 'bg_1', 'estatus_busqueda', 'bg_2', 'fub', 'DC_id', 'fecha_desap']

Primera Fecha y última fecha:
82335    01/01/1967
10573    31/12/2022
Name: fecha_desap, dtype: str


,Unnamed: 0,nombrecompleto,rnpdno_json,entidad_desap,pre_rnped,rnped,rnpdno_csv,bg_1,estatus_busqueda,bg_2,fub,DC_id,fecha_desap
0,1,LUIS ADRIAN JUAREZ MORALES,1,QUERETARO,0,0,1,0,NaN,0,NaN,"101 (json), 5673 (csv)",19/04/2023
1,2,ERNESTO ALONSO HERNANDEZ GUTIERREZ,1,QUERETARO,0,0,1,1,Con indicios / Ubicada,1,187731B1B-3302-4BE3-81D5-018CFB86A00B,"349 (json), 448 (csv), 187731B1B-3302-4BE3-81D...",12/07/2016
2,3,PIEDAD JESUS GUERRERO CHAVEZ,1,QUERETARO,0,0,1,1,Se busca reportante,1,122009919-199C-4B90-8462-935A6EDCE439,"732 (json), 14618 (csv), 122009919-199C-4B90-8...",26/09/2017
3,4,LORENZO MARTINEZ,1,QUERETARO,0,0,1,1,Se requieren datos de identidad,1,148436E49-364B-4663-847C-0338E23B2460,"1007 (json), 89640 (csv), 148436E49-364B-4663-...",17/03/2010
4,5,AGUSTINA OLVERA OCHOA,1,QUERETARO,0,0,1,1,NaN,0,NaN,"1013 (json), 57259 (csv)",04/06/2023


En base a lo anterior podemos observar que:

1. El dataset tiene 89720 filas y 13 columnas
2. El periodo de fechas cubierto está en el rango entre 1 de enero de 1967 hasta el 31 de diciembre del 2022 *pero creo que hay un error con los métodos con los que obtuve esa información porque posteriormente en el head me salen registros del 2023*
3. Cada entrada en el dataset corresponde a la información de una persona desaparecida. Entre lo más importante se encuentra el nombre, la entidad donde desaperecio, el estado de búsqueda y la fecha de desaparición.

III. Medidas descriptivas y análisis exploratorio de datos (EDA)

1. Diccionario de datos.
   
Antes de pasar al diccionario de datos, es importante señalar el signficado de las siglas "rnped" y "rnpedno". Esto significa "Red Nacional de Personas Desaparecidas y No Localizadas." 

In [1]:
from __future__ import annotations
from typing import Dict

import pandas as pd

dataset_path = "tot_reg_desap.csv"


def infer_statistical_type(series: pd.Series) -> str:
    """
    Clasifica el tipo estadístico de una variable:
    - numérico
    - categórico
    """
    if pd.api.types.is_numeric_dtype(series):
        return "numérico"
    return "categórico"


def infer_semantic_type(series: pd.Series, col_name: str) -> str:
    """
    Regresa una etiqueta de tipo de dato más legible para reporte.
    """
    if pd.api.types.is_integer_dtype(series):
        return "entero"
    if pd.api.types.is_float_dtype(series):
        return "flotante"
    if pd.api.types.is_bool_dtype(series):
        return "booleano"
    if pd.api.types.is_datetime64_any_dtype(series):
        return "fecha/hora"

    # Intento de detección de fecha si la columna viene como texto
    if "fecha" in col_name.lower():
        parsed = pd.to_datetime(series, errors="coerce", dayfirst=True)
        if parsed.notna().mean() > 0.7:
            return "fecha (texto parseable)"

    return "texto"


def explain_column(col: str):
    """
    Descripción semántica por nombre de columna.
    """
    c = col.strip().lower()

    dictionary: Dict[str, str] = {
        "dataset_id": "Identificador único del registro.",
        "nombrecompleto": "Nombre completo de la persona reportada como desaparecida/no localizada.",
        "rnpdno_json": "Presencia/coincidencia en RNPDNO (0/1).",
        "entidad_desap": "Entidad donde se reporta la desaparición/no localización.",
        "pre_rnped": "Indicador previo asociado a RNPED (0/1), usado para cruce/validación entre fuentes.",
        "rnped": "Presencia/coincidencia con registros RNPED (0/1).",
        "rnpdno_csv": "Presencia/coincidencia en RNPDNO en formato CSV (0/1).",
        "bg_1": "Bandera/indicador auxiliar de consistencia o disponibilidad de información (primer indicador).",
        "estatus_busqueda": "Estatus del proceso de búsqueda.",
        "bg_2": "Bandera/indicador auxiliar de consistencia o disponibilidad de información (segundo indicador).",
        "fub": "Folio Único de Búsqueda.",
        "dc_id": "Identificador de cruce o conciliación entre distintas fuentes (JSON/CSV/FUB).",
        "fecha_desap": "Fecha de desaparición o no localización reportada.",
    }

    if c in dictionary:
        return dictionary[c]

    if "fecha" in c:
        return "Campo de fecha asociado al registro."
    if "id" in c:
        return "Identificador del registro o de conciliación entre fuentes."
    if "estatus" in c:
        return "Estado o categoría del proceso asociado al registro."
    if "entidad" in c:
        return "Entidad geográfica/federativa asociada al registro."
    if "nombre" in c:
        return "Nombre o texto identificador de persona."
    if c.startswith("bg_"):
        return "Bandera/indicador binario auxiliar."


def build_data_dictionary(df: pd.DataFrame) -> pd.DataFrame:
    rows = []

    for col in df.columns:
        series = df[col]
        non_null = series.dropna()
        n_unique = non_null.nunique()
        example_values = ", ".join(map(str, non_null.astype(str).unique()[:2]))
        semantic_type = infer_semantic_type(series, col)

        rows.append(
            {
                "Variable": col,
                "Descripcion": explain_column(col),
                "Tipo Dato (Pandas)": str(series.dtype),
                "Tipo Dato (Semantico)": semantic_type,
                "Tipo Estadistico": infer_statistical_type(series),
                "# No Nulos": int(series.notna().sum()),
                "# Nulos": int(series.isna().sum()),
                "# Únicos No Nulos": int(n_unique),
                "Ejemplos": example_values,
            }
        )

    return pd.DataFrame(rows)


df = pd.read_csv(dataset_path, low_memory=False)
# La primera columna del dataset no tiene un nombre asignado.
df = df.rename(columns={df.columns[0]: "dataset_id"})
data_dict = build_data_dictionary(df)
display(data_dict)

,Variable,Descripcion,Tipo Dato (Pandas),Tipo Dato (Semantico),Tipo Estadistico,# No Nulos,# Nulos,# Únicos No Nulos,Ejemplos
0,dataset_id,Identificador único del registro.,int64,entero,numérico,89720,0,89720,"1, 2"
1,nombrecompleto,Nombre completo de la persona reportada como d...,str,texto,categórico,89720,0,88513,"LUIS ADRIAN JUAREZ MORALES, ERNESTO ALONSO HER..."
2,rnpdno_json,Presencia/coincidencia en RNPDNO (0/1).,int64,entero,numérico,89720,0,2,"1, 0"
3,entidad_desap,Entidad donde se reporta la desaparición/no lo...,str,texto,categórico,89720,0,33,"QUERETARO, CHIHUAHUA"
4,pre_rnped,"Indicador previo asociado a RNPED (0/1), usado...",int64,entero,numérico,89720,0,2,"0, 1"
5,rnped,Presencia/coincidencia con registros RNPED (0/1).,int64,entero,numérico,89720,0,2,"0, 1"
6,rnpdno_csv,Presencia/coincidencia en RNPDNO en formato CS...,int64,entero,numérico,89720,0,2,"1, 0"
7,bg_1,Bandera/indicador auxiliar de consistencia o d...,int64,entero,numérico,89720,0,2,"0, 1"
8,estatus_busqueda,Estatus del proceso de búsqueda.,str,texto,categórico,62356,27364,9,"Con indicios / Ubicada, Se busca reportante"
9,bg_2,Bandera/indicador auxiliar de consistencia o d...,int64,entero,numérico,89720,0,2,"0, 1"


2. Aplicación de CRISP-DM al dataset asignado

3. Cálculo de medidas con visualizaciones e interpretación

   * Medidas de localización (media, moda, mediana)

     Para este análisis tomaremos en cuenta variables numéricas las cuales se puede sacar información importante como:
   * 'entidad_desap': (¿Cuál es la media y mediana de desaparecidos en cada entidad?, ¿Cuál entidad es la que más aparece (moda)?)
   * 'fecha_desap': (¿Cuál es la media y mediana de desapariciones por año?, ¿Cuál es el año con más desapariciones (moda)?)
   * 'estatus_busqueda': 

In [2]:
# Convertir fecha a datetime
df["fecha_desap"] = pd.to_datetime(
    df["fecha_desap"],
    format="%d/%m/%Y",
    errors="coerce"
)

# =========================
# ANALISIS: entidad_desap
# =========================

print("\n=========================")
print("VARIABLE: entidad_desap")
print("=========================")

# Frecuencias
freq_entidad = df["entidad_desap"].value_counts(dropna=False)
print("\nFrecuencias:")
print(freq_entidad.head(10))

# Moda
moda_entidad = df["entidad_desap"].mode()
print("\nModa:")
print(moda_entidad)

# =========================
# ANALISIS: estatus_busqueda
# =========================

print("\n=========================")
print("VARIABLE: estatus_busqueda")
print("=========================")

# Frecuencias
freq_estatus = df["estatus_busqueda"].value_counts(dropna=False)
print("\nFrecuencias:")
print(freq_estatus.head(10))

# Moda
moda_estatus = df["estatus_busqueda"].mode()
print("\nModa:")
print(moda_estatus)

# =========================
# ANALISIS: fecha_desap
# =========================

print("\n=========================")
print("VARIABLE: fecha_desap")
print("=========================")

# Estadísticos
fecha_min = df["fecha_desap"].min()
fecha_max = df["fecha_desap"].max()
fecha_mediana = df["fecha_desap"].median()

print("\nFecha mínima:", fecha_min)
print("Fecha máxima:", fecha_max)
print("Mediana:", fecha_mediana)

# =========================
# EXTRA (opcional)
# =========================

# Conteo por año
df["anio"] = df["fecha_desap"].dt.year
conteo_anual = df["anio"].value_counts().sort_index()

print("\nConteo por año:")
print(conteo_anual.tail(10))


VARIABLE: entidad_desap

Frecuencias:
entidad_desap
TAMAULIPAS          11897
ESTADO DE MEXICO    10963
VERACRUZ             6043
SINALOA              5426
MICHOACAN            5126
JALISCO              4709
NUEVO LEON           4622
SONORA               4340
CHIHUAHUA            4144
GUERRERO             4129
Name: count, dtype: int64

Moda:
0    TAMAULIPAS
Name: entidad_desap, dtype: str

VARIABLE: estatus_busqueda

Frecuencias:
estatus_busqueda
NaN                                                        27364
Se busca reportante                                        22863
Se requieren datos de identidad                            18425
Con indicios / Ubicada                                     12622
Denuncia confirmada                                         8408
Se requieren datos de identidad, Se busca reportante          20
Se requieren datos de identidad, Con indicios / Ubicada        8
Con indicios / Ubicada, Se requieren datos de identidad        4
Se busca reportante, Se req